In [8]:
# Exploration on samples catalogue in databricks
from databricks.connect import DatabricksSession
import os
from dotenv import load_dotenv
import pyarrow.dataset as ds
import pyarrow.fs as pafs
import time
from databricks.sdk.runtime import dbutils
import duckdb
import gc

load_dotenv()


True

In [9]:
import gc


gc.collect()

0

In [10]:

if os.getenv("DATABRICKS_RUNTIME_VERSION"):  # running ON Databricks
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
else:  # local dev via Databricks Connect
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.remote(
        host=os.getenv("DATABRICKS_HOST"),
        token=os.getenv("DATABRICKS_TOKEN")
    ).serverless(True).getOrCreate()

# spark.conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
# spark.conf.set("fs.s3a.endpoint.region", "us-east-1")

In [11]:
spark.read?


Type:        property
String form: <property object at 0x117ce8810>
Docstring:  
Returns a :class:`DataFrameReader` that can be used to read data
in as a :class:`DataFrame`.

.. versionadded:: 2.0.0

.. versionchanged:: 3.4.0
    Supports Spark Connect.

Returns
-------
:class:`DataFrameReader`

Examples
--------
>>> spark.read
<...DataFrameReader object ...>

Write a DataFrame into a JSON file and read it back.

>>> import tempfile
>>> with tempfile.TemporaryDirectory(prefix="read") as d:
...     # Write a DataFrame into a JSON file
...     spark.createDataFrame(
...         [{"age": 100, "name": "Hyukjin Kwon"}]
...     ).write.mode("overwrite").format("json").save(d)
...
...     # Read the JSON file as a DataFrame.
...     spark.read.format('json').load(d).show()
+---+------------+
|age|        name|
+---+------------+
|100|Hyukjin Kwon|
+---+------------+

In [5]:

# public s3 scan base url
S3_BASE_URL = "s3://aws-public-blockchain/v1.0/btc"

# pulling sample of tx data from btc

# scan urls for transactions and blocks
scan_date_prefix="2026-06-*"  # for testing, scan all of 2026
btc_tx_scan_url = f'{S3_BASE_URL}/transactions/date={scan_date_prefix}/*.parquet'
btc_block_scan_url = f'{S3_BASE_URL}/blocks/date={scan_date_prefix}/*.parquet'



In [6]:
df = spark.read.parquet(f"{S3_BASE_URL}/transactions/")
df_filtered = df.where("date = '2026-06-04'")
# df_filtered.select( "txid", "input_count").show(15)




In [7]:
df_filtered.select("txid", "input_count").explain(True)

== Parsed Logical Plan ==
'Project ['txid, 'input_count]
+- 'Filter ('date = 2026-06-04)
   +- Relation [txid#11178,hash#11179,version#11180L,size#11181L,block_hash#11182,block_number#11183L,index#11184L,virtual_size#11185L,lock_time#11186L,input_count#11187L,output_count#11188L,is_coinbase#11189,output_value#11190,outputs#11191,block_timestamp#11192,date#11193,last_modified#11194] parquet

== Analyzed Logical Plan ==
txid: string, input_count: bigint
Project [txid#11178, input_count#11187L]
+- Filter (date#11193 = cast(2026-06-04 as date))
   +- Relation [txid#11178,hash#11179,version#11180L,size#11181L,block_hash#11182,block_number#11183L,index#11184L,virtual_size#11185L,lock_time#11186L,input_count#11187L,output_count#11188L,is_coinbase#11189,output_value#11190,outputs#11191,block_timestamp#11192,date#11193,last_modified#11194] parquet

== Optimized Logical Plan ==
Project [txid#11178, input_count#11187L]
+- Filter (isnotnull(date#11193) AND (date#11193 = 2026-06-04))
   +- Relation

In [10]:
spark.sql("SHOW CATALOGS").show()


+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



In [11]:
spark.sql("SHOW SCHEMAS IN workspace").show()

+------------------+
|      databaseName|
+------------------+
|          btc_data|
|           default|
|information_schema|
+------------------+



In [12]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.btc_data")

DataFrame[]

In [13]:
# list tables in btc_data schema
spark.sql("SHOW TABLES IN workspace.btc_data").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|btc_data|   blocks|      false|
+--------+---------+-----------+



In [22]:
#querying blocks table from btc_data schema using the dataframe api
_blk = spark.read.table("workspace.btc_data.blocks")

# size of _blk dataframe in bytes
_blk.collect()

[Row(hash='00000000035b69b78d5a96e2e1a62e9a110b18876b0c714d047935ab44a4d5ed', version=1, mediantime=datetime.datetime(2010, 7, 12, 19, 47, 29), nonce=7480340, bits='1c0ae493', difficulty=23.50125722290317, chainwork='0000000000000000000000000000000000000000000000000004ee73b5b6fc1d', size=374, weight=1496, coinbase_param='0493e40a1c026504', number=66208, transaction_count=2, merkle_root='cbe824b4e6c1fc10cdb1ff2a66910d53a132f7ccc9c08126bbbd1626c072501d', stripped_size=374, timestamp=datetime.datetime(2010, 7, 12, 20, 0, 14), date=datetime.date(2010, 7, 13), last_modified=datetime.datetime(2026, 2, 6, 13, 27, 21, 475749)),
 Row(hash='000000000530061f9d47ee0d8515d12450d5d842d435102d20bcdde8f1a5f669', version=1, mediantime=datetime.datetime(2010, 7, 12, 19, 53, 8), nonce=8476465, bits='1c0ae493', difficulty=23.50125722290317, chainwork='0000000000000000000000000000000000000000000000000004ee8b3620e13a', size=374, weight=1496, coinbase_param='0493e40a1c027b02', number=66209, transaction_count

In [ ]:
spark.sql("SELECT * FROM workspace.btc_data.blocks").show(5)



+--------------------+-------+-------------------+----------+--------+----------+--------------------+----+------+--------------+------+-----------------+--------------------+-------------+-------------------+----------+--------------------+
|                hash|version|         mediantime|     nonce|    bits|difficulty|           chainwork|size|weight|coinbase_param|number|transaction_count|         merkle_root|stripped_size|          timestamp|      date|       last_modified|
+--------------------+-------+-------------------+----------+--------+----------+--------------------+----+------+--------------+------+-----------------+--------------------+-------------+-------------------+----------+--------------------+
|00000000839a8e688...|      1|2009-01-09 02:54:25|2573394689|1d00ffff|       1.0|00000000000000000...| 215|   860|04ffff001d0104|     1|                1|0e3e2357e806b6cdb...|          215|2009-01-09 02:54:25|2009-01-09|2026-02-06 17:25:...|
|000000008d9dc510f...|      1|20

In [28]:
s3 = pafs.S3FileSystem(anonymous=True, region="us-east-2")

dataset = ds.dataset(
    "aws-public-blockchain/v1.0/btc/transactions/",
    filesystem=s3,
    partitioning="hive"
)

In [29]:
partitions = dataset.get_fragments()


In [30]:
list(partitions)[-10:]

[<pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/btc/transactions/date=2026-08-16/962767.snappy.parquet partition=[date=2026-08-16]>,
 <pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/btc/transactions/date=2026-08-16/962768.snappy.parquet partition=[date=2026-08-16]>,
 <pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/btc/transactions/date=2026-08-16/962769.snappy.parquet partition=[date=2026-08-16]>,
 <pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/btc/transactions/date=2026-08-16/962770.snappy.parquet partition=[date=2026-08-16]>,
 <pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/btc/transactions/date=2026-08-16/962771.snappy.parquet partition=[date=2026-08-16]>,
 <pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/btc/transactions/date=2026-08-16/962772.snappy.parquet partition=[date=2026-08-16]>,
 <pyarrow.dataset.ParquetFileFragment path=aws-public-blockchain/v1.0/

In [31]:
fragments = dataset.get_fragments(filter=(ds.field("date") >= "2009-01-01") & (ds.field("date") <= "2026-06-30"))
paths = sorted({f.path for f in fragments})
spark_paths = [f"s3://{p.rsplit('/', 1)[0]}/" for p in paths]

In [ ]:
df = (
    spark.read
    .option("basePath", "s3://aws-public-blockchain/v1.0/btc/transactions")
    .parquet(*spark_paths)
)

'aws-public-blockchain/v1.0/btc/transactions/date=2009-01-09/part-00000-d26a2cfd-6464-4c75-872d-fbaf830b9ff9-c000.snappy.parquet'

In [34]:
paths[:2]

['aws-public-blockchain/v1.0/btc/transactions/date=2009-01-03/part-00000-0b2f1dc9-9663-4a45-b566-d390f3e31b15-c000.snappy.parquet',
 'aws-public-blockchain/v1.0/btc/transactions/date=2009-01-09/part-00000-d26a2cfd-6464-4c75-872d-fbaf830b9ff9-c000.snappy.parquet']

In [35]:
df1 = df.where("date = '2026-01-01'")
df1.drop('inputs','outputs')


DataFrame[txid: string, hash: string, version: bigint, size: bigint, block_hash: string, block_number: bigint, index: bigint, virtual_size: bigint, lock_time: bigint, input_count: bigint, output_count: bigint, is_coinbase: boolean, output_value: double, block_timestamp: timestamp, date: date, last_modified: timestamp]

In [36]:
data = df1.collect()

In [ ]:
# load data to a dataframe
_df = spark.createDataFrame(data)


DataFrame[txid: string, hash: string, version: bigint, size: bigint, block_hash: string, block_number: bigint, index: bigint, virtual_size: bigint, lock_time: bigint, input_count: bigint, output_count: bigint, is_coinbase: boolean, output_value: double, outputs: array<struct<index:bigint,required_signatures:bigint,script_asm:string,script_hex:string,type:string,value:double>>, block_timestamp: timestamp, date: date, last_modified: timestamp]

In [39]:
_df.show(5)

+--------------------+--------------------+-------+----+--------------------+------------+-----+------------+---------+-----------+------------+-----------+--------------------+--------------------+-------------------+----------+--------------------+
|                txid|                hash|version|size|          block_hash|block_number|index|virtual_size|lock_time|input_count|output_count|is_coinbase|        output_value|             outputs|    block_timestamp|      date|       last_modified|
+--------------------+--------------------+-------+----+--------------------+------------+-----+------------+---------+-----------+------------+-----------+--------------------+--------------------+-------------------+----------+--------------------+
|10720a90eee1a4a74...|10720a90eee1a4a74...|      2| 223|00000000000000000...|      930486|  148|         223|        0|          1|           2|      false|          0.06848117|[{0, 1, 0 140b340...|2026-01-01 23:08:23|2026-01-01|2026-02-12 04:19:.

In [ ]:
# -- 1. Schema (already created, idempotent)
# CREATE SCHEMA IF NOT EXISTS workspace.btc_data;

# -- 2. External table pointing at S3 (no data copy, just metadata)
# CREATE TABLE workspace.btc_data.tx_raw
# USING PARQUET
# PARTITIONED BY (date)
# LOCATION 's3://aws-public-blockchain/v1.0/btc/transactions/';

# -- 3. One-time crawl: registers every date partition in Unity Catalog
# ALTER TABLE workspace.btc_data.tx_raw RECOVER PARTITIONS;

# -- 4. Verify partitions discovered
# SHOW PARTITIONS workspace.btc_data.tx_raw LIMIT 20;

# -- 5. Pure partition-pruned query — should be instant after step 3
# SELECT date, COUNT(*) AS tx_count
# FROM workspace.btc_data.tx_raw
# WHERE date = '2026-06-04'
# GROUP BY date;

In [61]:
spark.sql("show schemas in workspace").show()

+------------------+
|      databaseName|
+------------------+
|          btc_data|
|           default|
|information_schema|
+------------------+



In [49]:
df.select("txid","block_number").show(10)

+--------------------+------------+
|                txid|block_number|
+--------------------+------------+
|f562923477656a56c...|      782123|
|6cb95c5e1313af1ab...|      782123|
|ef76e7b4bcc4316e5...|      782123|
|b7d1a71e9f69a874a...|      782123|
|d5dc80736d5fa1cd7...|      782123|
|668c6f480f58ea3ad...|      782123|
|67e475f4e8e0a663f...|      782123|
|64f8a3c9f6b9b06cb...|      782123|
|56851e3b119c3d5cb...|      782123|
|8f4fc63082008b134...|      782123|
+--------------------+------------+
only showing top 10 rows


In [29]:
df.explain(mode='formatted')

== Physical Plan ==
LocalTableScan (1)


(1) LocalTableScan
Output [17]: [txid#11180, hash#11181, version#11182L, size#11183L, block_hash#11184, block_number#11185L, index#11186L, virtual_size#11187L, lock_time#11188L, input_count#11189L, output_count#11190L, is_coinbase#11191, output_value#11192, outputs#11193, block_timestamp#11194, date#11195, last_modified#11196]
Arguments: <empty>, [txid#11180, hash#11181, version#11182L, size#11183L, block_hash#11184, block_number#11185L, index#11186L, virtual_size#11187L, lock_time#11188L, input_count#11189L, output_count#11190L, is_coinbase#11191, output_value#11192, outputs#11193, block_timestamp#11194, date#11195, last_modified#11196]


== Photon Explanation ==
Photon does not fully support the query because:
		Unsupported node: LocalTableScan <empty>, [txid#11180, hash#11181, version#11182L, size#11183L, block_hash#11184, block_number#11185L, index#11186L, virtual_size#11187L, lock_time#11188L, input_count#11189L, output_count#11190L, is_coi

In [41]:
spark_paths = [f"s3://{p.rsplit('/', 1)[0]}/" for p in paths]  # back to partition-folder level
df = spark.read.parquet(*set(spark_paths))

In [42]:
df_filtered = df.where("date = '2021-01-04'")


In [43]:
df_filtered.select("txid",'input_count').show()

+--------------------+-----------+
|                txid|input_count|
+--------------------+-----------+
|bd1a2ae5fdd3adb5b...|          0|
|f52a735ff587ea191...|          1|
|e09da7afae62fc321...|          1|
|07fa9bf3ed749dc13...|          1|
|8d181afc0eb3c9215...|          1|
|e073f9e12df6258fb...|          1|
|900650aadd4657530...|          1|
|9270c54c24429ad5e...|          1|
|7c529c13e603c7e3d...|          1|
|13d9fe1f9c9f8d1d3...|          1|
|7fb7c7411a0d4e3fd...|          3|
|3f8f4a8c2faa2fca6...|          1|
|4153b158fab1f18da...|          1|
|32b08052d0a858d12...|          1|
|59cce098456a4c2bd...|          1|
|dfcd6650a7768c806...|          1|
|8c5e7d87d33d42c37...|          1|
|01698ad780e926bac...|          1|
|5f3f34f038dc3c38a...|          1|
|f8f6db18adf0497d8...|          2|
+--------------------+-----------+
only showing top 20 rows


In [47]:
q = """

WITH daily_stats AS (
    SELECT
        date,
        COUNT(*) AS tx_count,
        SUM(fee) AS total_fees,
        SUM(output_value) AS total_volume,
        AVG(fee) AS avg_fee,
        AVG(fee / NULLIF(virtual_size, 0)) AS avg_fee_per_vbyte,
        PERCENTILE_APPROX(fee, 0.5) AS median_fee
    FROM tx
    WHERE is_coinbase = false
      AND fee > 0
      AND date >= '2026-06-01'
    GROUP BY date
),
ranked AS (
    SELECT
        date,
        txid,
        fee,
        ROW_NUMBER() OVER (PARTITION BY date ORDER BY fee DESC) AS fee_rank,
        ROW_NUMBER() OVER (PARTITION BY date ORDER BY virtual_size DESC) AS size_rank
    FROM tx
    WHERE is_coinbase = false
      AND fee > 0
      AND date >= '2026-06-01'
),
top_tx AS (
    SELECT date, txid, fee
    FROM ranked
    WHERE fee_rank = 1
)
SELECT
    d.date,
    d.tx_count,
    ROUND(d.total_fees, 2) AS total_fees_btc,
    ROUND(d.total_volume, 0) AS total_volume_btc,
    ROUND(d.avg_fee, 8) AS avg_fee_btc,
    ROUND(d.avg_fee_per_vbyte * 1e8, 2) AS avg_sats_per_vbyte,
    ROUND(d.median_fee * 1e8, 0) AS median_fee_sats,
    t.txid AS highest_fee_txid,
    ROUND(t.fee, 8) AS highest_fee_btc,
    ROUND(SUM(d.total_fees) OVER (ORDER BY d.date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS cumulative_fees
FROM daily_stats d
JOIN top_tx t ON d.date = t.date
ORDER BY d.date
"""
# Filter to June only, register as temp view
tx_df = df.where("date >= '2026-06-01' AND date <= '2026-06-30'")
tx_df.createOrReplaceTempView("tx")

# Verify
spark.sql("SELECT COUNT(*), MIN(date), MAX(date) FROM tx").show()


+--------+----------+----------+
|COUNT(*)| MIN(date)| MAX(date)|
+--------+----------+----------+
|19549661|2026-06-01|2026-06-30|
+--------+----------+----------+



In [49]:
sts= spark.sql(q)

In [ ]:
sts.collect()

In [109]:


blocks_query = """
SELECT *
FROM parquet.`s3://aws-public-blockchain/v1.0/btc/blocks/`
WHERE date >= '2026-06-01'
"""

blocks_df = spark.sql(blocks_query)


In [111]:
blocks_df.count()

9358

# Duckdb

In [44]:
con = duckdb.connect()

# 1. Load the httpfs extension to enable S3 access
con.execute("INSTALL httpfs; LOAD httpfs;")

# 2. Tell DuckDB which region the bucket is in
con.execute("SET s3_region='us-east-2';")


In [48]:
query = f"""

select txid, input_count

from {S3_BASE_URL}/transactions/

where date = '2026-06-04'

limit 1000
"""
con.sql(query)

ParserException: Parser Error: syntax error at or near "//"

LINE 5: from s3://aws-public-blockchain/v1.0/btc/transactions/
                ^

In [47]:
S3_BASE_URL

's3://aws-public-blockchain/v1.0/btc'